In [30]:
import os
import pandas as pd
import numpy as np
import datetime

In [31]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [32]:
path = "/content/drive/MyDrive/Math168/Data/"
print(os.listdir(path))

['Finalized Data', 'tx_birank_priors_test1m.csv', 'tx_birank_priors_train.csv', 'feat_test1m.csv', 'feat_train.csv', 'Extracted_Fraud_Test_Data 2.csv', 'Extracted fraud train data.csv', 'apate_birank_features_train.csv', 'apate_birank_features_test1m.csv', 'apate_features_train.csv', 'apate_features_test1m.csv', 'distance_train.csv', 'distance_test_1m.csv', 'updated_merged_train.csv', 'updated_merged_test.csv']


In [33]:
documents_train = ['feat_train.csv', 'distance_train.csv', 'apate_features_train.csv', 'apate_birank_features_train.csv', 'tx_birank_priors_train.csv',]
documents_test = ['feat_test1m.csv', 'distance_test_1m.csv', 'apate_features_test1m.csv', 'apate_birank_features_test1m.csv', 'tx_birank_priors_test1m.csv',]

In [34]:
path2 = "/content/drive/MyDrive/Math168/fraud_detection/"
print(os.listdir(path2))

['fraudTrain.csv', 'fraudTest.csv', 'tx_birank_priors.csv', 'feat_train.csv', 'feat_test1m.csv', 'tx_birank_priors_train.csv', 'tx_birank_priors_test1m.csv', 'card_birank.csv', 'merchant_birank.csv', 'apate_features.csv', 'apate_features_birank.csv']


In [35]:
fraudTest = pd.read_csv(path2 + "fraudTest.csv", index_col=0)
fraudTrain = pd.read_csv(path2 + "fraudTrain.csv", index_col=0)

In [36]:
fraudTest['trans_date_trans_time'] = pd.to_datetime(fraudTest['trans_date_trans_time'], errors='coerce')

In [37]:
test_start = fraudTest['trans_date_trans_time'].min()
test_end = test_start + pd.DateOffset(months=1)   # first calendar month
fraudTest_1m = fraudTest[
    (fraudTest['trans_date_trans_time'] >= test_start) &
    (fraudTest['trans_date_trans_time'] < test_end)
]

In [38]:
def merge_selected_features(
    base_df,
    feature_file,
    selected_cols,
    path,
    id_col="trans_num",
    suffix=None
):
    df = pd.read_csv(path + feature_file)

    if id_col not in df.columns:
        df = df.reset_index()

    keep_cols = [id_col] + selected_cols
    df_small = df[keep_cols].copy()

    if suffix is not None:
        rename_map = {col: f"{col}_{suffix}" for col in selected_cols}
        df_small = df_small.rename(columns=rename_map)

    return base_df.merge(df_small, on=id_col, how="left")

### Merge training features

In [39]:
features_train = fraudTrain[
    ["trans_num", "amt", "lat", "long", "city_pop", "category", "unix_time", "merch_lat", "merch_long", "is_fraud"]
].copy()

In [40]:
feat1 = ['log_amt', 'merchant_txn_count_past', 'merchant_unique_cards_past', 'card_txn_count_1h','amt_to_card_median', 'amt_to_category_median', 'hour_of_day', 'day_of_week', 'is_weekend', 'dist_home_merchant_km', 'recency_sec', 'dist_prev_merchant_km', 'implied_speed_kmh']
feat2 = ['State Fraud Rate', 'Avg Card Dist (km)', 'Dist Ratio', 'Log Distance (km)', 'Bin Fraud Rate']
feat3 = ['CCHScore_ST', 'MCScore_ST', 'TXScore_ST', 'CCHScore_MT', 'MCScore_MT', 'TXScore_MT', 'CCHScore_LT', 'MCScore_LT', 'TXScore_LT']
feat4 = ['card_birank', 'merchant_birank']

In [41]:
feat_cols = [feat1, feat2, feat3, feat3, feat4]

In [42]:
for i in range(len(documents_train)):
  suffix = None
  if i == 3:
    suffix = "birank"
  features_train = merge_selected_features(
      features_train,
      documents_train[i],
      feat_cols[i],
      path,
      suffix = suffix
      )

In [43]:
features_train = features_train.reindex()

In [44]:
features_train_path = path + "/Finalized Data/features_train.csv"

In [45]:
features_train.to_csv(features_train_path, index=True)

In [46]:
from google.colab import files
files.download(features_train_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Merge Testing Features

In [47]:
features_test = fraudTest_1m[
    ["trans_num", "amt", "lat", "long", "city_pop", "category", "unix_time", "merch_lat", "merch_long", "is_fraud"]
].copy()

In [48]:
for i in range(len(documents_test)):
  suffix = None
  if i == 3:
    suffix = "birank"
  features_test = merge_selected_features(
      features_test,
      documents_test[i],
      feat_cols[i],
      path,
      suffix = suffix
      )

In [49]:
features_test = features_test.reindex()

In [50]:
features_test

,trans_num,amt,lat,long,city_pop,category,unix_time,merch_lat,merch_long,is_fraud,...,MCScore_ST_birank,TXScore_ST_birank,CCHScore_MT_birank,MCScore_MT_birank,TXScore_MT_birank,CCHScore_LT_birank,MCScore_LT_birank,TXScore_LT_birank,card_birank,merchant_birank
0,2da90c7d74bd46a0caf3777415b3ebd3,2.86,33.9659,-80.9355,333497,personal_care,1371816865,33.986391,-81.200714,0,...,0.000536,0.000512,0.000166,0.000096,0.000014,0.000140,0.000117,0.000002,0.001176,0.001392
1,324cc204407e99f51b0d6ca0055005e7,29.84,40.3207,-110.4360,302,personal_care,1371816873,39.450498,-109.960431,0,...,0.000349,0.001081,0.000239,0.000139,0.000013,0.000246,0.000203,0.000003,0.001285,0.001455
2,c81755dbbbea9d5c77f094348a7579be,41.28,40.6729,-73.5365,34496,health_fitness,1371816893,40.495810,-74.196111,0,...,0.000185,0.000394,0.000182,0.000070,0.000011,0.000326,0.000175,0.000003,0.001588,0.001447
3,2159175b9efe66dc301f149d3d5abf8c,60.05,28.5697,-80.8191,54767,misc_pos,1371816915,28.812398,-80.883061,0,...,0.000476,0.000653,0.000148,0.001310,0.000066,0.000148,0.000186,0.000003,0.001142,0.001340
4,57ff021bd3f328f8738bb535c302a31b,3.19,44.2529,-85.0170,1126,travel,1371816917,44.959148,-85.884734,0,...,0.000117,0.000466,0.000100,0.000054,0.000011,0.000220,0.000130,0.000004,0.001308,0.000916
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87640,28dbfcfd6288b406e125154d26a8048b,102.10,40.5046,-77.7186,4653,grocery_pos,1374408807,41.103110,-77.601875,0,...,0.000350,0.000880,0.000201,0.000252,0.000015,0.000246,0.001569,0.000011,0.001269,0.001873
87641,a057b7eecf1683d1109afd59d6478d6a,7.95,29.3641,-98.4924,1595797,personal_care,1374408810,29.993934,-98.163099,0,...,0.000454,0.001105,0.000180,0.000183,0.000014,0.000294,0.000217,0.000004,0.001400,0.001383
87642,ba8f31627906ee914d3150b3557ebe03,123.11,26.1184,-81.7361,276002,home,1374408812,25.973459,-81.848113,0,...,0.000396,0.000809,0.000108,0.000238,0.000015,0.000170,0.000298,0.000003,0.001134,0.001659
87643,014019287afe680e698414b3ca38dc09,811.45,38.5072,-81.8900,5512,misc_net,1374408816,37.583384,-82.006009,0,...,0.000139,0.000928,0.000403,0.000156,0.000020,0.000281,0.000499,0.000007,0.001433,0.001146


In [51]:
features_test_path = path + "/Finalized Data/features_test.csv"

In [52]:
features_test.to_csv(features_test_path, index=True)

In [53]:
from google.colab import files
files.download(features_test_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>